# GO term and GO enrichement

In [ ]:
# Builds GO-ready background & hit gene sets keyed by Fjoh_#### using an existing PPV-filtered PANNZER GO file, and writes mismatch reports (missing IDs / missing GO) for later cleanup.

import pandas as pd

# ---- INPUT FILES ----
go_filt_in = "GO.filtered_PPV0.3.txt"  # existing filtered GO file
all_quant_in = 'All Quantified Genes.tsv'        # RNA-seq: all quantified genes (background source)
de_in = "DE genes rnaseq.csv"                    # RNA-seq: significant genes (hit source)

# ---- LOAD FILTERED GO ----
go_filt = pd.read_csv(go_filt_in, sep="\t")
go_genes = set(go_filt["qpid"].dropna().astype(str).unique())

# ---- LOAD RNA-seq FILES ----
allq = pd.read_csv(all_quant_in, sep="\t")
de = pd.read_csv(de_in)

# column names
bg_col = "Locustag"
hit_col = "Locus tag"
alt_col = "Alt locus tag"
geneid_col = "Gene ID"

# ---- BACKGROUND SET (Fjoh_#### present, and GO-annotated) ----
bg_all = set(allq[bg_col].dropna().astype(str).unique())
background = bg_all & go_genes

# ---- HIT SET (significant genes file; Fjoh_#### present, and in background) ----
hit_all = set(de[hit_col].dropna().astype(str).unique())
hits = hit_all & background

# ---- REPORTS ----
# 1) Significant genes lacking primary Fjoh_#### ID (the old and new locus tag saga from NCBI/IMG)
de_missing_primary = de[de[hit_col].isna()].copy()
cols_keep = [c for c in [alt_col, geneid_col, "Gene", "Description", "FeatureType", "logFC", "FDR"] if c in de_missing_primary.columns]
de_missing_primary_out = "REPORT.DE_missing_primary_Fjoh_id.csv"
de_missing_primary[cols_keep].to_csv(de_missing_primary_out, index=False)

# 2) Significant genes with Fjoh_#### but no GO (after PPV filtering)
de_with_primary = de[de[hit_col].notna()].copy()
de_with_primary[hit_col] = de_with_primary[hit_col].astype(str)
de_no_go = de_with_primary[~de_with_primary[hit_col].isin(go_genes)].copy()
de_no_go_out = "REPORT.DE_with_primary_but_no_GO.csv"
cols_keep2 = [c for c in [hit_col, alt_col, geneid_col, "Gene", "Description", "FeatureType", "logFC", "FDR"] if c in de_no_go.columns]
de_no_go[cols_keep2].to_csv(de_no_go_out, index=False)

# 3) Background genes missing GO (coverage)
bg_no_go = sorted(bg_all - go_genes)
pd.Series(bg_no_go, name="Locustag").to_csv("REPORT.BACKGROUND_primary_but_no_GO.txt", index=False)

# 4) Final sets for enrichment
pd.Series(sorted(background), name="Locustag").to_csv("SET.background_Fjoh_GO_annotated.txt", index=False)
pd.Series(sorted(hits), name="Locustag").to_csv("SET.hits_Fjoh_GO_annotated.txt", index=False)

# ---- QUICK SUMMARY ----
print(f"GO-annotated genes (filtered file): {len(go_genes):,}")
print(f"All quantified genes (primary IDs): {len(bg_all):,}")
print(f"Background used (quantified ∩ GO): {len(background):,}")
print(f"DE genes (primary IDs present): {len(hit_all):,}")
print(f"Hits used (DE ∩ background): {len(hits):,}\n")

print("Wrote:")
print(" -", de_missing_primary_out)
print(" -", de_no_go_out)
print(" - REPORT.BACKGROUND_primary_but_no_GO.txt")
print(" - SET.background_Fjoh_GO_annotated.txt")
print(" - SET.hits_Fjoh_GO_annotated.txt")

In [ ]:
# Runs GO term enrichment (Fisher's exact test + Benjamini–Hochberg FDR) using Fjoh_#### IDs: hits = significant genes ∩ background, background = all quantified genes ∩ GO-annotated genes.

import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

# ---- INPUTS  ----
go_filt_in = "GO.filtered_PPV0.3.txt"  # existing filtered GO file
all_quant_in = 'All Quantified Genes.tsv'        # RNA-seq: all quantified genes (background source)
de_in = "DE genes rnaseq.csv"                    # RNA-seq: significant genes (hit source)

# ---- SETTINGS ----
min_go_genes = 5          # drop tiny GO terms
max_go_genes = 500        # drop overly broad GO terms (adjust if needed)
top_n_plot   = 20         # number of terms to plot
out_tsv      = "GO_enrichment_results.tsv"

# ---- LOAD ----
go = pd.read_csv(go_filt_in, sep="\t", dtype={"qpid": str, "goid": str, "ontology": str, "desc": str})
allq = pd.read_csv(all_quant_in, sep="\t")
de   = pd.read_csv(de_in)

# ---- BUILD SETS ----
bg_col  = "Locustag"
hit_col = "Locus tag"

go_genes = set(go["qpid"].dropna().unique())
bg_all   = set(allq[bg_col].dropna().astype(str).unique())
background = bg_all & go_genes

hit_all = set(de[hit_col].dropna().astype(str).unique())
hits = hit_all & background

print(f"GO-annotated genes: {len(go_genes):,}")
print(f"Quantified genes (primary IDs): {len(bg_all):,}")
print(f"Background used (quantified ∩ GO): {len(background):,}")
print(f"Hits used (DE ∩ background): {len(hits):,}")

if len(hits) == 0:
    raise ValueError("No hits overlap the GO-annotated background; check ID matching/columns.")

# ---- TERM -> GENESET ----
# Keep only GO rows for genes in background
go_bg = go[go["qpid"].isin(background)].copy()

term_to_genes = go_bg.groupby(["goid", "ontology", "desc"])["qpid"].apply(lambda s: set(s)).to_dict()

N = len(background)
K = len(hits)

rows = []
for (goid, ont, desc), geneset in term_to_genes.items():
    n = len(geneset)
    if n < min_go_genes or n > max_go_genes:
        continue

    a = len(geneset & hits)          # in hit set AND has term
    if a == 0:
        continue                     # no enrichment signal possible

    b = K - a                        # in hit set, not term
    c = n - a                        # in background, not hit, has term
    d = (N - n) - b                  # in background, not hit, not term

    # one-sided Fisher: enrichment (odds ratio > 1)
    odds, p = fisher_exact([[a, b], [c, d]], alternative="greater")

    rows.append({
        "goid": goid,
        "ontology": ont,
        "term": desc,
        "hit_with_term": a,
        "hit_total": K,
        "bg_with_term": n,
        "bg_total": N,
        "odds_ratio": odds,
        "p_value": p,
        "hit_fraction": a / K,
        "bg_fraction": n / N,
        "enrichment": (a / K) / (n / N) if (n / N) > 0 else np.nan,
        "genes_in_hits": ",".join(sorted(geneset & hits)),
    })

res = pd.DataFrame(rows)
if res.empty:
    raise ValueError("No GO terms passed filters with any hit genes; try lowering min_go_genes or PPV cutoff.")

# ---- BH-FDR ----
res = res.sort_values("p_value").reset_index(drop=True)
m = len(res)
res["fdr_bh"] = (res["p_value"] * (np.arange(1, m + 1) / 1.0) ** -1)  # temporary, overwritten below

# Correct BH implementation
rank = np.arange(1, m + 1)
q = res["p_value"].values * m / rank
q = np.minimum.accumulate(q[::-1])[::-1]
res["fdr_bh"] = np.clip(q, 0, 1)

# Save results
res = res.sort_values(["fdr_bh", "p_value", "enrichment"], ascending=[True, True, False])
res.to_csv(out_tsv, sep="\t", index=False)
print(f"\nSaved enrichment table: {out_tsv}")
print("Top 10 terms (by FDR):")
print(res[["goid","ontology","term","hit_with_term","bg_with_term","enrichment","p_value","fdr_bh"]].head(10).to_string(index=False))

# ---- QUICK PLOT (top terms by -log10(FDR)) ----
plot_df = res.head(top_n_plot).copy()
plot_df["neglog10_fdr"] = -np.log10(np.maximum(plot_df["fdr_bh"].values, 1e-300))

plt.figure(figsize=(10, 0.4 * len(plot_df) + 2))
plt.barh(plot_df["term"][::-1], plot_df["neglog10_fdr"][::-1])
plt.xlabel("-log10(FDR)")
plt.title(f"Top {min(top_n_plot, len(plot_df))} enriched GO terms (PPV-filtered)")
plt.tight_layout()
plt.show()

In [ ]:
# splitting the output file above into BP and MF then they will go to REVIGO to remove redundancy


# Splits enrichment results into BP and MF (drops CC), formats GO IDs as GO:#########, and writes two REVIGO input files using FDR values.

import pandas as pd

infile = "GO_enrichment_results.tsv"

res = pd.read_csv(infile, sep="\t", dtype={"goid": str, "ontology": str})
res = res[res["ontology"].isin(["BP", "MF"])].copy()

# format GO IDs as GO:0000000
res["goid"] = res["goid"].str.strip()
res["goid"] = res["goid"].apply(lambda x: x if x.startswith("GO:") else f"GO:{x.zfill(7)}")

# keep best (smallest) FDR per GO term
res = res.dropna(subset=["fdr_bh"])
res = res.groupby(["ontology", "goid"], as_index=False)["fdr_bh"].min()

# REVIGO expects two columns: GO_ID <tab> score (p or FDR); usually no header
bp = res[res["ontology"] == "BP"][["goid", "fdr_bh"]]
mf = res[res["ontology"] == "MF"][["goid", "fdr_bh"]]

bp_out = "REVIGO_BP.txt"
mf_out = "REVIGO_MF.txt"

bp.to_csv(bp_out, sep="\t", index=False, header=False)
mf.to_csv(mf_out, sep="\t", index=False, header=False)

print("Wrote:", bp_out, "(", len(bp), "terms )")
print("Wrote:", mf_out, "(", len(mf), "terms )")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load data
df = pd.read_csv("significant_MF_for_dot_plot.csv")

# Filter by FDR
df = df[df['fdr_bh'] < 0.05]

# Extract term names (before underscore)
df['term_short'] = df['term'].str.split('_').str[0]

#------
# Compute GeneRatio

# Column names in your CSV (EDIT if needed)
TERM_COL = "term"
HIT_COL  = "hit_with_term"
TOT_COL  = "hit_total"
FDR_COL  = "fdr_bh" 


df[HIT_COL] = pd.to_numeric(df[HIT_COL], errors="coerce").astype(int)
df[TOT_COL] = pd.to_numeric(df[TOT_COL], errors="coerce").astype(int)
df["GeneRatio"] = df[HIT_COL] / df[TOT_COL]


#------

# Sort by enrichment or p-value
df = df.sort_values('GeneRatio', ascending=True)

# Plot parameters
fig_width = 4
fig_height = len(df) * 1.5  # Adjust multiplier for spacing

# Create figure
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

# Create scatter plot
scatter = ax.scatter(
    df['GeneRatio'],
    range(len(df)),
    # c=-np.log10(df['fdr_bh'])
    c=df['fdr_bh'],
    s=df['hit_with_term'] * 15,  # Adjust multiplier for dot size
    cmap='jet',
    edgecolors='lightgray',
    linewidth=0.5,
    alpha=0.8
)

# Set y-axis
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['term_short'], fontname='Helvetica', fontsize=10)

# Set x-axis
ax.set_xlabel('GeneRatio', fontname='Helvetica', fontsize=11, fontweight='bold')
ax.set_xlim(left=0, right=0.05)

# Title
ax.set_title('Gene Set Enrichment Analysis', 
             fontname='Helvetica', fontsize=12, fontweight='bold', pad=15)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('FDR', fontname='Helvetica', fontsize=10, fontweight='bold')
cbar.ax.tick_params(labelsize=9)

# Style
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelsize=9)
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('enrichment_dotplot.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# fig fpor paper

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np

# Load data
df = pd.read_csv("significant_MF_for_dot_plot.csv")

# Filter by FDR
df = df[df['fdr_bh'] < 0.05]

# Extract term names (before underscore)
df['term_short'] = df['term'].str.split('_').str[0]

#------
# Compute GeneRatio
TERM_COL = "term"
HIT_COL  = "hit_with_term"
TOT_COL  = "hit_total"
FDR_COL  = "fdr_bh"

df[HIT_COL] = pd.to_numeric(df[HIT_COL], errors="coerce").astype(int)
df[TOT_COL] = pd.to_numeric(df[TOT_COL], errors="coerce").astype(int)
df["GeneRatio"] = df[HIT_COL] / df[TOT_COL]
#------

# Sort by enrichment or p-value
df = df.sort_values('GeneRatio', ascending=True)

# Plot parameters
fig_width = 4.3
fig_height = len(df) * 1  # Adjust multiplier for spacing

# Normalization: option 1 — anchor at 0, vmax at FDR cutoff
# Low FDR (most significant) → darkest with Greys_r
fdr_cutoff = 0.05
norm = Normalize(vmin=0.0, vmax=fdr_cutoff)

# Create figure
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

# Create scatter plot
scatter = ax.scatter(
    df['GeneRatio'],
    range(len(df)),
    c=df['fdr_bh'],
    s=df['hit_with_term'] * 30,  # Adjust multiplier for dot size
    cmap='Greys_r',              # Inverted: low FDR = dark, high FDR = light
    edgecolors='black',          # Black edge so light dots remain visible
    linewidth=0.5,
    alpha=1,
    norm=norm
)

# Set y-axis
ax.set_ylim(-0.2, len(df) - 0.7)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['term_short'], fontname='Helvetica', fontsize=14)

# Set x-axis
ax.set_xlabel('GeneRatio', fontname='Helvetica', fontsize=14, fontweight='bold')
ax.set_xlim(left=0, right=0.061)
ax.set_xticks(np.arange(0, 0.062, 0.03))
ax.tick_params(axis='x', labelsize=13)

# Title
ax.set_title('Gene Set Enrichment Analysis',
             fontname='Helvetica', fontsize=12, fontweight='bold', pad=15)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('FDR', fontname='Helvetica', fontsize=14, fontweight='bold')
cbar.ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig('enrichment_dotplot_grayscale_600dpi.png', dpi=600, bbox_inches='tight')
plt.show()


# MAKING A CNETPLOT (GENE CONCEPT NETWORK PLOT) NOW


In [ ]:
# Creates Cytoscape input files for a Gene–Concept Network from your MF enrichment table:
# (1) edges: TERM -> GENE, (2) nodes: attributes for styling (type, FDR, GeneRatio, counts, labels).

import pandas as pd
import numpy as np

infile = "significant_MF_for_dot_plot.csv"

# -------------------- WHAT TO INCLUDE (EDIT THESE) --------------------
use_top_n = 12            # set None to ignore
fdr_cutoff = 0.05         # set None to ignore
out_edges = "cnet_edges_MF.tsv"
out_nodes = "cnet_nodes_MF.tsv"

# -------------------- LOAD --------------------
df = pd.read_csv(infile)

# -------------------- FILTER TERMS --------------------
df["fdr_bh"] = pd.to_numeric(df["fdr_bh"], errors="coerce")
df = df.dropna(subset=["fdr_bh"]).copy()

if fdr_cutoff is not None:
    df = df[df["fdr_bh"] <= fdr_cutoff].copy()

df = df.sort_values(["fdr_bh", "p_value", "hit_with_term"], ascending=[True, True, False]).copy()

if use_top_n is not None:
    df = df.head(use_top_n).copy()

if df.empty:
    raise ValueError("No terms left after filtering; loosen fdr_cutoff or increase use_top_n.")

# -------------------- LABELS --------------------
# You asked: use text before '_' only for display
df["term_label"] = df["term"].astype(str).str.split("_", n=1).str[0]

# Safer unique IDs for Cytoscape (avoid collisions if two labels are same)
df["term_id"] = "TERM:" + df["term"].astype(str)

# GeneRatio (for node attributes)
df["hit_with_term"] = df["hit_with_term"].astype(int)
df["hit_total"] = df["hit_total"].astype(int)
df["GeneRatio"] = df["hit_with_term"] / df["hit_total"]

# -------------------- BUILD EDGE LIST --------------------
edges = []
for _, r in df.iterrows():
    term_id = r["term_id"]
    genes = str(r["genes_in_hits"]).split(",")
    genes = [g.strip() for g in genes if g.strip()]
    for g in genes:
        edges.append({
            "source": term_id,
            "target": "GENE:" + g,
            "interaction": "annotates"
        })

edges = pd.DataFrame(edges).drop_duplicates()

# -------------------- BUILD NODE TABLE --------------------
# Term nodes
term_nodes = df[["term_id", "term_label", "fdr_bh", "p_value", "hit_with_term", "bg_with_term", "GeneRatio"]].copy()
term_nodes = term_nodes.rename(columns={"term_id": "id", "term_label": "label"})
term_nodes["type"] = "GO_term"

# Gene nodes
gene_ids = sorted(set(edges["target"]))
gene_nodes = pd.DataFrame({
    "id": gene_ids,
    "label": [g.replace("GENE:", "") for g in gene_ids],
    "type": "gene"
})

# Combine
nodes = pd.concat([term_nodes, gene_nodes], ignore_index=True)

# -------------------- WRITE FILES --------------------
edges.to_csv(out_edges, sep="\t", index=False)
nodes.to_csv(out_nodes, sep="\t", index=False)

print("Wrote:", out_edges, f"({len(edges)} edges)")
print("Wrote:", out_nodes, f"({len(nodes)} nodes)")
print("Terms included:", len(df))

In [ ]:
# using cytoscape to make the network 

# RNASEQ volcano plot

volcano plot

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION - 
# ═══════════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────────
# 1. FILE PATHS & DATA COLUMNS
# ──────────────────────────────────────────────────────────────────────────────
INPUT_FILE = '/All Quantified Genes.tsv'
OUTPUT_FILE = "volcano_plot1.png"
SEPARATOR = "\t"  # "\t" for TSV, "," for CSV

# Column names in the data file
LOCUS_COL = "Locustag"
GENE_COL = "Gene"
DESC_COL = "Description"
LOGFC_COL = "logFC"
FDR_COL = "FDR"

# ──────────────────────────────────────────────────────────────────────────────
# 2. SIGNIFICANCE THRESHOLDS
# ──────────────────────────────────────────────────────────────────────────────
FDR_THRESHOLD = 0.05
LOGFC_THRESHOLD = 1  # |logFC| >= 1 means 2-fold change

# ──────────────────────────────────────────────────────────────────────────────
# 3. PLOT APPEARANCE
# ──────────────────────────────────────────────────────────────────────────────
FIGURE_SIZE = (8, 5)
XLIM = None   
YLIM = None  

# Background points (non-significant genes)
BG_COLOR = "#D3D3D3"  # Light gray
BG_SIZE = 50
BG_ALPHA = 0.15

# Significant genes (not highlighted by GO terms)
UP_COLOR = "#E67E22"      
DOWN_COLOR = "#16A085"    
SIG_SIZE = 75
SIG_ALPHA = 0.3

# Threshold lines
SHOW_THRESHOLD_LINES = True
THRESHOLD_LINE_COLOR = "darkgrey"
THRESHOLD_LINE_STYLE = "--"
THRESHOLD_LINE_WIDTH = 0.5

# ──────────────────────────────────────────────────────────────────────────────
# 4. GO TERM GENE SETS - Defining the highlight groups here
# ──────────────────────────────────────────────────────────────────────────────



def parse_gene_string(s):
    return [x.strip() for x in str(s).split(",") if x.strip()]
# gene lists here
    # moved to the cell above

    
GO_HIGHLIGHTS = {
    "Siderophore uptake": {
        "genes": parse_gene_string(sid_genes_top5),
        "style": "fill",
        "color": "#C0392B",        
        "size_multiplier": 1.2,
        "alpha": 0.95,
        "zorder": 6,
    },
    
    "Signaling receptor": {
        "genes": parse_gene_string(sign_genes_top5),
        "style": "outline",
        "color": "dimgray",       
        "size_multiplier": 1.2,
        "outline_width": 1.7,
        "outline_gap": 2.5,        # Distance from fill edge
        "gap_width": 2.0,          # White gap thickness
        "alpha": 1.0,
        "zorder": 9,
    },
    
    "Heme binding": {
        "genes": parse_gene_string(heme_genes_top5 ),
        "style": "fill",
        "color": "mediumorchid",        
        "size_multiplier": 1.2,
        "alpha": 0.95,
        "zorder": 7,
    },
    
    "Electron transfer": {
        "genes": parse_gene_string(et_genes_top5),
        "style": "outline",
        "color": "blue",        
        "size_multiplier": 1.2,
        "outline_width": 1.5,
        "outline_gap": 2.5,
        "gap_width": 2.0,
        "alpha": 1.0,
        "zorder": 10,
    },
}

# Base size for all GO highlights (adjust this to scale everything)
GO_BASE_SIZE = 75

# ──────────────────────────────────────────────────────────────────────────────
# 5. OPTIONAL TEXT LABELS
# ──────────────────────────────────────────────────────────────────────────────
SHOW_LABELS = False  # Set to True to show labels
LABELS_TO_SHOW = []
LABEL_FONTSIZE = 7


# ═══════════════════════════════════════════════════════════════════════════════
# FUNCTIONS - 
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set publication-quality fonts
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})


def load_and_prepare_data(filepath, sep, logfc_col, fdr_col, fdr_thr, logfc_thr):
    """Load data and calculate significance categories."""
    df = pd.read_csv(filepath, sep=sep)
    
    # Convert to numeric and clean
    df[logfc_col] = pd.to_numeric(df[logfc_col], errors="coerce")
    df[fdr_col] = pd.to_numeric(df[fdr_col], errors="coerce")
    df = df.dropna(subset=[logfc_col, fdr_col]).copy()
    df = df[df[fdr_col] > 0].copy()
    
    # Calculate -log10(FDR)
    df["neglog10_fdr"] = -np.log10(df[fdr_col])
    
    # Assign significance categories
    df["sig"] = "ns"
    df.loc[(df[fdr_col] <= fdr_thr) & (df[logfc_col] >= logfc_thr), "sig"] = "up"
    df.loc[(df[fdr_col] <= fdr_thr) & (df[logfc_col] <= -logfc_thr), "sig"] = "down"
    
    return df


def plot_background_points(ax, df, x_col, y_col):
    """Plot non-significant, up, and down-regulated genes."""
    ns = df[df["sig"] == "ns"]
    up = df[df["sig"] == "up"]
    down = df[df["sig"] == "down"]
    
    ax.scatter(ns[x_col], ns[y_col], c=BG_COLOR, s=BG_SIZE, 
               alpha=BG_ALPHA, linewidths=0.01, edgecolors="black")
    ax.scatter(down[x_col], down[y_col], c=DOWN_COLOR, s=SIG_SIZE, 
               alpha=SIG_ALPHA, linewidths=0, edgecolors="none")
    ax.scatter(up[x_col], up[y_col], c=UP_COLOR, s=SIG_SIZE, 
               alpha=SIG_ALPHA, linewidths=0, edgecolors="none")
    
    return len(up), len(down), len(ns)


def plot_go_highlights(ax, df, go_dict, x_col, y_col, locus_col, base_size):
    """Plot GO term highlights (fills and outlines)."""
    for term_name, spec in go_dict.items():
        genes = spec.get("genes", [])
        if not genes:
            continue
        
        sub = df[df[locus_col].isin(genes)]
        if sub.empty:
            continue
        
        style = spec.get("style", "fill")
        color = spec.get("color", "red")
        size = base_size * spec.get("size_multiplier", 1.0)
        alpha = spec.get("alpha", 1.0)
        z = spec.get("zorder", 8)
        
        if style == "outline":
            # Draw outline with gap
            gap_mult = spec.get("outline_gap", 2.0)
            gap_width = spec.get("gap_width", 2.0)
            outline_width = spec.get("outline_width", 1.5)
            
            # White gap ring
            ax.scatter(sub[x_col], sub[y_col], s=size * gap_mult,
                      facecolors="none", edgecolors="white",
                      linewidths=gap_width, alpha=1.0, zorder=z, clip_on=False)
            
            # Colored outline ring
            ax.scatter(sub[x_col], sub[y_col], s=size * gap_mult,
                      facecolors="none", edgecolors=color,
                      linewidths=outline_width, alpha=alpha, zorder=z+1, clip_on=False)
        
        else:  # fill
            ax.scatter(sub[x_col], sub[y_col], s=size, c=color,
                      edgecolors=color, linewidths=0, alpha=alpha, 
                      zorder=z, clip_on=False)


def add_labels(ax, df, locus_tags, x_col, y_col, locus_col, fontsize):
    """Add non-overlapping text labels using adjustText."""
    try:
        from adjustText import adjust_text
    except ImportError:
        print("Warning: adjustText not installed. Install with: pip install adjustText")
        return
    
    if not locus_tags:
        return
    
    sub = df[df[locus_col].isin(locus_tags)].copy()
    if sub.empty:
        print("No matching locus tags found to annotate.")
        return
    
    texts = []
    for _, row in sub.iterrows():
        texts.append(
            ax.text(row[x_col], row[y_col], str(row[locus_col]),
                   fontsize=fontsize, ha="left", va="bottom")
        )
    
    adjust_text(texts, ax=ax, expand_text=(1.1, 1.2), expand_points=(1.1, 1.2),
                force_text=(0.4, 0.6), force_points=(0.2, 0.3), only_move={"text": "xy"})


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PLOTTING CODE
# ═══════════════════════════════════════════════════════════════════════════════

# Load data
df = load_and_prepare_data(INPUT_FILE, SEPARATOR, LOGFC_COL, FDR_COL, 
                           FDR_THRESHOLD, LOGFC_THRESHOLD)

# Create plot
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

# Plot background points
n_up, n_down, n_ns = plot_background_points(ax, df, LOGFC_COL, "neglog10_fdr")

# Add GO highlights
plot_go_highlights(ax, df, GO_HIGHLIGHTS, LOGFC_COL, "neglog10_fdr", 
                   LOCUS_COL, GO_BASE_SIZE)

# Add threshold lines
if SHOW_THRESHOLD_LINES:
    ax.axvline(LOGFC_THRESHOLD - 0.1, color=THRESHOLD_LINE_COLOR, 
               linestyle=THRESHOLD_LINE_STYLE, linewidth=THRESHOLD_LINE_WIDTH)
    ax.axvline(-LOGFC_THRESHOLD + 0.1, color=THRESHOLD_LINE_COLOR, 
               linestyle=THRESHOLD_LINE_STYLE, linewidth=THRESHOLD_LINE_WIDTH)
    ax.axhline(-np.log10(FDR_THRESHOLD), color=THRESHOLD_LINE_COLOR, 
               linestyle=THRESHOLD_LINE_STYLE, linewidth=THRESHOLD_LINE_WIDTH)

# Add labels if requested
if SHOW_LABELS:
    add_labels(ax, df, LABELS_TO_SHOW, LOGFC_COL, "neglog10_fdr", 
               LOCUS_COL, LABEL_FONTSIZE)

# Set labels and limits
ax.set_xlabel("log2 Fold Change")
ax.set_ylabel("-log10(FDR)")

if XLIM is not None:
    ax.set_xlim(XLIM)
if YLIM is not None:
    ax.set_ylim(YLIM)
else:
    ax.set_ylim(bottom=0)

# Finalize
fig.tight_layout()
fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")  # Uncomment to save
plt.show()

# Print summary
print(f"Total genes: {len(df)}")
print(f"Upregulated: {n_up}")
print(f"Downregulated: {n_down}")
print(f"Non-significant: {n_ns}")